In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
!pip install openpyxl
import openpyxl

import warnings
warnings.filterwarnings("ignore")

In [ ]:
tickers = pd.read_excel('nifty500_universe_classified.xlsx', index_col=0)

In [ ]:
# input
start = '2012-01-01' 
end = '2026-01-01' 

In [ ]:
print("Parsing Started....")
count=0
for i in tickers.Symbol:
  symbol = i+ '.NS'
  df = yf.download(symbol,start,end)
  df.to_csv("NIFTY500/"+str(count)+ "_" + i +'.csv')
  print("Done ", count, " ", symbol)
  count+=1

  # if count==3:
  #   break
  print("PARSED!")

In [ ]:
import yfinance as yf
import pandas as pd

rows = []

for t in tickers:
    try:
        info = yf.Ticker(t).info

        rows.append({
            "ticker": t,
            "company_name": info.get("longName", "Unknown"),
            "sector": info.get("sector", "Unknown"),
            "market_cap": info.get("marketCap", None)
        })

        print("Done:", t)

    except Exception as e:
        print("Failed:", t, e)

In [ ]:

import os
import glob

# Define your input and output directories
input_folder = "NIFTY500"
output_folder = "dataset500"

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Find all CSV files in the input folder
csv_files = glob.glob(f"{input_folder}/*.csv")

print(f"Found {len(csv_files)} files. Starting cleanup...\n")

for file in csv_files:
    filename = os.path.basename(file)
    output_path = os.path.join(output_folder, filename)
    
    try:
        # yfinance saves CSVs with a 2-level header (Price, Ticker)
        # header=[0, 1] reads the first two rows as column names
        # index_col=0 sets the first column ('Date') as the index
        df = pd.read_csv(file, header=[0, 1], index_col=0)
        
        # Flatten the MultiIndex columns to just the first level (Open, High, etc.)
        # This removes the 'AARTIIND.NS' ticker level completely
        df.columns = df.columns.get_level_values(0)
        
        # Ensure the index is named 'Date'
        df.index.name = 'date'
        
        # Define the desired OHLCV column order
        # (We check against df.columns to avoid errors if a column is missing)
        desired_order = ['Open', 'High', 'Low', 'Close', 'Volume']
        final_columns = [col for col in desired_order if col in df.columns]
        
        # Reorder the dataframe
        df = df[final_columns]
        
        # Drop rows where all OHLCV values are NaN (cleans up any empty artifact rows)
        df = df.dropna(how='all')
        
        # Save to the clean folder
        df.to_csv(output_path)
        print(f"Cleaned and saved: {filename}")
        
    except Exception as e:
        print(f"ERROR cleaning {filename}: {e}")

print("\nCleanup Completed.")

In [ ]:
import os
import shutil


BASE_DATASET = "dataset500"
OUT_DIR = "classified_dataset"

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Size folders
for size in ["large", "mid", "small"]:
    os.makedirs(os.path.join(OUT_DIR, "size", size), exist_ok=True)

# Sector folders
for sector in df["sector"].dropna().unique():
    os.makedirs(os.path.join(OUT_DIR, "sector", sector), exist_ok=True)

def clean_sector(sector):
    return sector.replace("/", "_").strip()

for _, row in df.iterrows():
    ticker = row["ticker"]
    size = row["size"]
    sector = row["sector"]
    clean_sector(sector)
    src = os.path.join(BASE_DATASET, f"{ticker}.csv")

    if not os.path.exists(src):
        continue

    # Copy to size folder
    dst_size = os.path.join(OUT_DIR, "size", size, f"{ticker}.csv")
    shutil.copy2(src, dst_size)

    # Copy to sector folder
    if pd.notna(sector):
        dst_sector = os.path.join(OUT_DIR, "sector", sector, f"{ticker}.csv")
        shutil.copy2(src, dst_sector)

In [ ]:
```python
"""
daily_snapshot_loader.py
=======================

Loads OHLCV data from CSV files, enriches with metadata,
cleans it, converts into a date-wise daily snapshot,
and saves a single parquet file.

Output:
data_complete/daily_snapshot.parquet
"""

import os
import json
import logging
from datetime import datetime

import pandas as pd


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR = "dataset500"
META_FILE = "nifty500_universe_classified.xlsx"

DATA_ROOT = "data_complete"
OUTPUT_FILE = os.path.join(DATA_ROOT, "combined_full_clean_ordered.parquet")


# ─────────────────────────────────────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────────────────────────────────────

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


# ─────────────────────────────────────────────────────────────────────────────
# LOAD METADATA
# ─────────────────────────────────────────────────────────────────────────────

def load_metadata():
    df = pd.read_excel(META_FILE)
    df.columns = [c.strip() for c in df.columns]

    meta = df.set_index("Ticker").to_dict("index")
    return meta, df


# ─────────────────────────────────────────────────────────────────────────────
# LOAD CSV DATA
# ─────────────────────────────────────────────────────────────────────────────

def load_all_csv():
    frames = []

    files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
    logger.info(f"Loading {len(files)} CSV files...")

    for f in files:
        try:
            path = os.path.join(DATA_DIR, f)

            ticker = f.split("_")[-1].replace(".csv", "") + ".NS"

            df = pd.read_csv(path)

            if df.empty:
                continue

            df["date"] = pd.to_datetime(df["date"])

            df = df.rename(columns={
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close",
                "Adj Close": "adj_close",
                "Volume": "volume"
            })

            df["ticker"] = ticker

            frames.append(
                df[["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]]
            )

        except Exception as e:
            logger.warning(f"Failed loading {f}: {e}")

    panel = pd.concat(frames, ignore_index=True)

    logger.info(f"Loaded panel shape: {panel.shape}")
    return panel


# ─────────────────────────────────────────────────────────────────────────────
# CLEANING
# ─────────────────────────────────────────────────────────────────────────────

def clean_panel(panel):
    logger.info("Cleaning panel...")

    # Remove invalid prices
    panel = panel[(panel[["open", "high", "low", "close"]] > 0).all(axis=1)]

    # Sort for return calc
    panel = panel.sort_values(["ticker", "date"])

    # Remove extreme returns
    panel["ret"] = panel.groupby("ticker")["close"].pct_change()
    panel = panel[panel["ret"].abs() < 0.8]

    panel = panel.drop(columns=["ret"])

    # Forward fill per ticker
    panel = panel.groupby("ticker").apply(lambda x: x.ffill(limit=3))
    panel = panel.reset_index(drop=True)

    # Volume fix
    panel["volume"] = panel["volume"].fillna(0)

    logger.info("Cleaning complete.")
    return panel


# ─────────────────────────────────────────────────────────────────────────────
# METADATA ENRICHMENT
# ─────────────────────────────────────────────────────────────────────────────

def add_metadata(panel, meta_dict):
    logger.info("Adding metadata...")

    panel["sector"] = panel["ticker"].map(
        lambda t: meta_dict.get(t, {}).get("Sector", "Unknown")
    )

    panel["cap"] = panel["ticker"].map(
        lambda t: meta_dict.get(t, {}).get("Cap", "Unknown")
    )

    return panel


# ─────────────────────────────────────────────────────────────────────────────
# BUILD DAILY SNAPSHOT
# ─────────────────────────────────────────────────────────────────────────────

def build_daily_snapshot(panel):
    logger.info("Building date-wise daily snapshot...")

    snapshot = panel.pivot(
        index="date",
        columns="ticker",
        values=["open", "high", "low", "close", "adj_close", "volume"]
    )

    # Flatten column MultiIndex
    snapshot.columns = [
        f"{ticker}_{field}" for field, ticker in snapshot.columns
    ]

    snapshot = snapshot.sort_index()

    logger.info(f"Snapshot shape: {snapshot.shape}")
    return snapshot


# ─────────────────────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────────────────────

def save_snapshot(snapshot, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    snapshot.to_parquet(path)

    meta = {
        "rows": int(len(snapshot)),
        "columns": int(snapshot.shape[1]),
        "start": str(snapshot.index.min().date()),
        "end": str(snapshot.index.max().date()),
        "generated": datetime.utcnow().isoformat(),
    }

    with open(path.replace(".parquet", "_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    logger.info(f"Saved snapshot: {path}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    logger.info("=" * 60)
    logger.info("DAILY SNAPSHOT PIPELINE")
    logger.info("=" * 60)

    meta_dict, _ = load_metadata()

    panel = load_all_csv()
    panel = clean_panel(panel)
    panel = add_metadata(panel, meta_dict)

    snapshot = build_daily_snapshot(panel)

    save_snapshot(snapshot, OUTPUT_FILE)

    logger.info("PIPELINE COMPLETE")


if __name__ == "__main__":
    main()
```
